# 02 - Baseline Model (LR + XGBoost)
Trains and evaluates baseline models, logs metrics to MLflow, and produces SHAP summary output.

In [ ]:
from pathlib import Path
import os
import sys

import joblib
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
import shap

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from falabella_risk.models.train_baseline import run_training

run_training(
    feature_path=Path('data/processed/features.parquet'),
    model_out=Path('models/baseline_xgb.pkl'),
    random_state=42,
    mlflow_uri=None,
    experiment_name='falabella_baseline_model',
)

In [ ]:
exp = mlflow.get_experiment_by_name('falabella_baseline_model')
runs = mlflow.search_runs([exp.experiment_id], order_by=['start_time DESC'])
runs[[
    'run_id',
    'metrics.lr_auc', 'metrics.xgb_auc',
    'metrics.lr_pr_auc', 'metrics.xgb_pr_auc',
    'metrics.lr_f1', 'metrics.xgb_f1',
    'metrics.lr_brier', 'metrics.xgb_brier',
]].head(1)

In [ ]:
model = joblib.load('models/baseline_xgb.pkl')
features = pd.read_parquet('data/processed/features.parquet')
x = features.drop(columns=['borrower_id', 'default_flag'], errors='ignore')

sample = x.sample(min(1500, len(x)), random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

plt.figure(figsize=(10, 6))
shap.summary_plot(sv, sample, show=False)
plt.tight_layout()
plt.show()